## Trading a Delta Hedged Straddle

1.  Buy 10 contracts of puts and calls at the strike closest to current stock price every morning
    a. Expiry that is closest to today but at least 5 trading days out.
    b. Hedge remaining delta
2.  Re-hedge if residual delta > 2 contracts
2.  Exit the trade at EOD.
5.  We assume no slippage (i.e. entry and exit at mid price) and commission of 1/2 a cent for stock trades and 1 dollar for option trades

First lets generate some option prices based on the underlying stock price.  We will generate some random volatility numbers, and use Black Scholes to generate prices.  We will add all this data to a strategy context that we can later use from our strategy and avoid excessive global variables.

In [ ]:
import numpy as np
import polars as pl
from dataclasses import dataclass
from typing import Sequence
import gambit as pq
from types import SimpleNamespace


_logger = pq.get_child_logger(__name__)


def get_symbol(put_call: str, strike: int, expiry: np.datetime64) -> str:
    '''Create option symbol from parameters'''
    return f'{put_call}-{strike}-{expiry}'


def parse_symbol(symbol: str) -> tuple[str, int, np.datetime64]:
    '''Break down option symbol into put_call, strike, expiry'''
    split = symbol.split('-', maxsplit=2)
    return (split[0], int(split[1]), np.datetime64(split[2]))


@dataclass
class StraddleEntryRule:
    expiries: dict[np.datetime64, np.ndarray]
    strikes: dict[tuple[np.datetime64, str, np.datetime64], np.ndarray]
    umids: dict[np.datetime64, float]
        
    def __call__(self,
                 contract_group: pq.ContractGroup,
                 i: int,
                 timestamps: np.ndarray,
                 indicator_values: SimpleNamespace,
                 signal_values: np.ndarray,
                 account: pq.Account,
                 current_orders: Sequence[pq.Order],
                 strategy_context: pq.StrategyContextType) -> list[pq.Order]:
        timestamp = timestamps[i]
        date = timestamp.astype('M8[D]')
        expiries = self.expiries[date]
        expiry = expiries[pq.np_find_closest(expiries, date + np.timedelta64(30, 'D'))]
        put_strikes = self.strikes[(date, 'P', expiry)]
        call_strikes = self.strikes[(date, 'C', expiry)]
        umid = self.umids[timestamp]
        cidx = pq.np_find_closest(call_strikes, umid)
        pidx = pq.np_find_closest(put_strikes, umid)
        found = False
        for j in range(10):
            put_strike = put_strikes[pidx]
            call_strike = call_strikes[cidx]
            put_symbol = get_symbol('P', put_strike, expiry)
            call_symbol = get_symbol('C', call_strike, expiry)
            if pq.Contract.exists(put_symbol):
                put_contract = pq.Contract.get(put_symbol)
            else:
                put_contract = pq.Contract.create(put_symbol, contract_group, expiry, 100)
            if pq.Contract.exists(call_symbol):
                call_contract = pq.Contract.get(call_symbol)
            else:
                call_contract = pq.Contract.create(call_symbol, contract_group, expiry, 100)
            put_delta = context.get_delta(put_contract, timestamps, i, context)
            call_delta = context.get_delta(call_contract, timestamps, i, context)
            if np.isfinite(put_delta) and np.isfinite(call_delta): 
                found = True
                break
            cidx += 1
            pidx -= 1
        if not found: return []
        symbol = f'{put_symbol}_{call_symbol}'
        if pq.Contract.exists(symbol):
            contract = pq.Contract.get(symbol)
        else:
            assert put_contract is not None
            assert call_contract is not None
            contract = pq.Contract.create(symbol, 
                                          contract_group,
                                          multiplier=100, 
                                          components=[(put_contract, 1.), (call_contract, 1.)])
        assert contract is not None
        order = pq.MarketOrder(contract=contract, timestamp=timestamp, qty=10, reason_code='ENTER_STRADDLE')
        _logger.info(f'ORDER: {timestamp} {order}')
        return [order]
    
    
@dataclass
class RehedgeRule:
    '''
    A rule to re-hedge deltas on a straddle
    Args:
        reason_code: the reason for the orders (used for display purposes)
        price_func: the function this rule uses to get market prices
    '''
    reason_code: str
    price_func: pq.PriceFunctionType
        
    def __init__(self, 
                 reason_code: str, 
                 price_func: pq.PriceFunctionType) -> None:
        self.reason_code = reason_code
        self.price_func = price_func
        
    def __call__(self,
                 contract_group: pq.ContractGroup,
                 i: int,
                 timestamps: np.ndarray,
                 indicator_values: SimpleNamespace,
                 signal_values: np.ndarray,
                 account: pq.Account,
                 current_orders: Sequence[pq.Order],
                 strategy_context: pq.StrategyContextType) -> list[pq.Order]:
        timestamp = timestamps[i]
        positions = account.positions(pq.ContractGroup.get('OPTIONS'), timestamp)
        
        orders: list[pq.Order] = []
        for (contract, qty) in positions:
            if not contract.is_basket(): continue
            put_contract, put_ratio = contract.components[0]
            call_contract, call_ratio = contract.components[1]
            put_qty = int(round(qty * call_ratio))
            call_qty = int(round(qty * put_ratio))
            hedge_contract, target_hedge_qty = get_hedge(
                put_contract, call_contract, put_qty, call_qty, timestamps, i, context)
            if np.isnan(target_hedge_qty): return []  # can have nan deltas sometimes
            hedge_positions = account.positions(contract_group, timestamp)
            pq.assert_(len(hedge_positions) in [0, 1], f'unexpected num of hedge positions: {hedge_positions}')
            curr_hedge_qty = 0
            if len(hedge_positions) == 1:
                curr_hedge_qty = int(round(hedge_positions[0][1]))
            hedge_qty = target_hedge_qty - curr_hedge_qty
            if hedge_qty == 0: return []
            hedge_order = pq.MarketOrder(contract=hedge_contract, 
                                         timestamp=timestamp, 
                                         qty=int(hedge_qty), 
                                         reason_code='REHEDGE')
            orders.append(hedge_order)
            _logger.info(f'ORDER: {timestamp} {hedge_order}')
        return orders
    

def get_hedge(put: pq.Contract, 
              call: pq.Contract, 
              put_qty: int, 
              call_qty: int, 
              timestamps: np.ndarray, 
              i: int, 
              context: pq.StrategyContextType) -> tuple[pq.Contract, int]:
    delta: float = 0
    delta += context.get_delta(put, timestamps, i, context) * put_qty
    delta += context.get_delta(call, timestamps, i, context) * call_qty
    if np.isnan(delta): delta = 0
    hedge_contract = pq.Contract.get('SPX')
    assert hedge_contract is not None
    hedge_qty = int(np.round(-100 * delta))
    return hedge_contract, hedge_qty


def get_expiries(prices: pl.DataFrame) -> dict[np.datetime64, np.ndarray]:
    _expiries = prices.select('date', 'expiry').sort(['date', 'expiry']).unique(maintain_order=True)
    dates = np.unique(_expiries['date'].to_numpy().astype('M8[D]'))
    expiry = _expiries['expiry'].to_numpy().astype('M8[D]')  
    expiries: dict[np.datetime64, np.ndarray] = {}
    for date in dates:
        expiries[date] = expiry[_expiries['date'].to_numpy() == date]
    return expiries


def get_strikes(prices: pl.DataFrame) -> dict[tuple[np.datetime64, str, np.datetime64], np.ndarray]:
    _strikes = prices.filter(pl.col('strike') % 100 == 0)
    keys = _strikes.select('date', 'put_call', 'expiry').sort(['date', 'put_call', 'expiry']).unique(maintain_order=True)
    _strikes = _strikes.select('date', 'put_call', 'expiry', 'strike').sort(['date', 'put_call', 'expiry', 'strike']).unique(maintain_order=True)
    date = keys['date'].to_numpy().astype('M8[D]')
    put_call = keys['put_call'].to_numpy()
    expiry = keys['expiry'].to_numpy().astype('M8[D]')
    strikes: dict[tuple[np.datetime64, str, np.datetime64], np.ndarray] = {}
    for i in range(len(date)):
        key = (date[i], put_call[i], expiry[i])
        _values = _strikes.filter((pl.col('date') == key[0]) & (pl.col('put_call') == key[1]) & (pl.col('expiry') == key[2]))
        values = _values['strike'].to_numpy().astype(int)
        strikes[key] = values
    return strikes


def get_price_function(prices: pl.DataFrame, field_name: str) -> pq.PriceFunctionType:
    price_dict: dict[str, tuple[np.ndarray, np.ndarray]] = {}
    for symbol in prices['symbol'].unique().to_list():
        sym_prc = prices.filter(pl.col('symbol') == symbol).select('timestamp', field_name).sort('timestamp')
        _timestamps = sym_prc['timestamp'].to_numpy().astype('M8[m]')
        _prices = sym_prc[field_name].to_numpy()
        price_dict[symbol] = (_timestamps, _prices)
    spx_prices = prices.select('timestamp', 'umid').sort('timestamp').unique(subset=['timestamp'], keep='first', maintain_order=True)
    price_dict['SPX'] = (spx_prices['timestamp'].to_numpy().astype('M8[m]'), spx_prices['umid'].to_numpy())
    return pq.PriceFuncArrayDict(price_dict=price_dict)


if __name__ == '__main__':
    pq.set_defaults()

    filename = pq.find_in_subdir('.', 'spx_options.csv.gz')
    prices = pl.read_csv(filename, try_parse_dates=True).select('timestamp', 'symbol', 'umid', 'c', 'delta')
    
    prices = prices.with_columns(pl.col('timestamp').dt.date().alias('date'))
    # prices = prices[prices['date'] == "2023-01-03"]

    # remove prices outside regular trading hours (9:30 am and 4 pm)
    minute = (prices['timestamp'].to_numpy() - prices['date'].to_numpy().astype('M8[us]')) / np.timedelta64(1, 'm')
    prices = prices.filter((minute > 9 * 60 + 30) & (minute < 16 * 60))

    prices = prices.with_columns(pl.col('symbol').str.splitn('-', 3).alias('parts')).unnest('parts').rename({'field_0': 'put_call', 'field_1': 'strike_text', 'field_2': 'expiry_text'}).with_columns(pl.col('strike_text').cast(pl.Int64).alias('strike'), pl.col('expiry_text').str.to_date('%Y-%m-%d').alias('expiry')).drop('strike_text', 'expiry_text')

    data = prices.select('timestamp', 'umid').sort('timestamp').unique(subset=['timestamp'], keep='first', maintain_order=True).with_columns(pl.col('timestamp').dt.date().alias('date'), pl.col('timestamp').dt.hour().alias('hour'))
    
    # Beginning and end of day and rehedge signals
    # Try 6 five-minute periods in case we don't get valid prices (every strike is not traded in every bar)
    data = data.with_columns((pl.col('date') != pl.col('date').shift(1)).fill_null(True).alias('bod_start')).with_columns(pl.col('bod_start').cast(pl.Int8).replace(0, None).forward_fill(limit=6).fill_null(0).cast(pl.Boolean).alias('bod')).drop('bod_start')
    
    # Try getting out 30 minutes before close so we have 6 bars to try and get out
    data = data.with_columns((pl.col('date') != pl.col('date').shift(-1)).fill_null(True).alias('eod_end')).with_columns(pl.col('eod_end').cast(pl.Int8).replace(0, None).backward_fill(limit=6).fill_null(0).cast(pl.Boolean).alias('eod')).drop('eod_end')
    
    data = data.with_columns((pl.col('hour') != pl.col('hour').shift(1)).fill_null(True).alias('rehedge'))
    
    strat_builder = pq.StrategyBuilder(data)
    context = pq.StrategyContextType()
    price_func = get_price_function(prices, 'c')
    delta_func = get_price_function(prices, 'delta')
    context.get_price = price_func
    context.get_delta = delta_func
    strat_builder.set_strategy_context(context)
    strat_builder.set_price_function(price_func)
    
    expiries = get_expiries(prices)
    strikes = get_strikes(prices)
    umids = {data['timestamp'].to_numpy()[i].astype('M8[m]'): data['umid'].to_numpy()[i] for i in range(len(data))}

    opt_cg = pq.ContractGroup.get('OPTIONS')
    strat_builder.add_contract_group(opt_cg)

    hedge_cg = pq.ContractGroup.get('HEDGES')
    spx = pq.Contract.create('SPX', contract_group=hedge_cg)
    strat_builder.add_contract_group(hedge_cg)

    # add rules to the strategy
    straddle_entry_rule = StraddleEntryRule(expiries, strikes, umids)
    strat_builder.add_series_rule('bod', straddle_entry_rule, position_filter='zero', contract_groups=[opt_cg])
    rehedge_rule = RehedgeRule('REHEDGE', context.get_price)
    strat_builder.add_series_rule('rehedge', rehedge_rule, contract_groups=[hedge_cg])
    close_rule = pq.ClosePositionExitRule('EOD', context.get_price)
    strat_builder.add_series_rule('eod', close_rule, position_filter='nonzero')

    # build and run strategy
    strategy = strat_builder()
    strategy.run()

In [ ]:
strategy.df_roundtrip_trades()

In [ ]:
strategy.evaluate_returns(plot=pq.has_display());

In [ ]:
# We can look at the peformance of our hedges separately
strategy.evaluate_returns(pq.ContractGroup.get('HEDGES'), plot=pq.has_display());